# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

Maximiliano Cencic

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [1]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi
from aima_libs.aima import PriorityQueue as AimaPriorityQueue

Función heurística basada en cantidad de discos (de mayor a menor) ubicados de forma correcta:
Ej:
- 3 discos correctos: h(x) = 3
- 4 discos corrects: h(x) = 4


In [ ]:
# Definimos una clase nueva que va a comprender info relevante del problema para cada nodo
class ProblemNode:
    """
    Clase que contiene información del problema para encolar y que sea 
    mas facil calcular la funcion heuristica

    Attributes:
        node (hanoi_states.NodeHanoi): El nodo actual del problema.
        goalRod (int): La torre objetivo
        disksNumber (int): Número de discos utilizados
    """
    def __init__(self, node: NodeHanoi, goalRod: int, disksNumber: int):
        self.node = node
        self.goalRod = goalRod
        self.disksNumber = disksNumber
    def get_node(self):
        return self.node
    def __lt__(self, other):
        return False

#Heuristica toma cantidad de discos bien colocados (loopea el poste para ver, según la cantidad de discos N, cuantos de ellos estan ok)
def h_priority_func(x: ProblemNode):
    rodList = x.node.state.get_state()
    stack = rodList[x.goalRod]
    count = 0
    for n in range(x.disksNumber): 
        if(len(stack) > n and stack[n] == x.disksNumber - n):
            count +=1
        else:
            break
    return ((count * -1) + x.node.state.get_accumulated_cost())
    

In [ ]:
def search_algorithm(number_disks=5) -> tuple[NodeHanoi, dict]:
    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    ##### EDITAR ESTA ZONA

    # Inicializamos las salidas, pero reemplazar con lo que se quiera usar.
    metrics = {
        "solution_found": False,
        "nodes_explored": 0,
        "states_visited": 0,
        "nodes_in_frontier": 1,
        "max_depth": 0,
        "cost_total": 0,
    }

    solution = NodeHanoi(initial_state)
    already_visited_hashes = set()
    h_priority_queue = AimaPriorityQueue(order='min', f=h_priority_func)
    h_priority_queue.append(ProblemNode(solution, 2, number_disks)) 

    while len(h_priority_queue) != 0:
        current_problem_queue_obj = h_priority_queue.pop()
        current_problem_node = current_problem_queue_obj[-1]
        metrics["nodes_explored"] += 1
        metrics["nodes_in_frontier"] -= 1

        if problem.goal_test(current_problem_node.get_node().state):
            metrics["states_visited"] += 1 
            metrics["solution_found"] = True
            metrics["cost_total"] = current_problem_node.get_node().state.get_accumulated_cost()
            solution = current_problem_node.get_node()
            break
        metrics["states_visited"] += 1
        for node in current_problem_node.get_node().expand(problem):
            if hash(node.state) not in already_visited_hashes:
                already_visited_hashes.add(hash(node.state))
                h_priority_queue.append(ProblemNode(node, 2, number_disks))
                metrics["nodes_in_frontier"] += 1
        metrics["max_depth"] = max(metrics["max_depth"], current_problem_node.get_node().depth)
    return [solution, metrics]

Se prueba la función:

In [4]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [5]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 169
states_visited: 169
nodes_in_frontier: 13
max_depth: 30
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [6]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [7]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
